<a href="https://colab.research.google.com/github/somendrew/LMS_Image_Classification/blob/main/LMS_Team_Query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Teams DataSet

In [1]:
import pandas as pd

In [2]:
teams = pd.read_csv("iteration_5_team.csv")

teams.shape

(602, 4)

In [9]:
teams

,Root,team_mid,team_name,team_name[1],testing_team_name,simplified_testing_team_name
0,/g/11nfsgpx7x,/m/03tcwm,Deportivo Alaves,NaN,Deportivo Alavés,Deportivo Alaves
1,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca,Alverca
2,/g/11nfsgpx7x,/m/03tcwm,Deportivo Alaves,NaN,Deportivo Alavés,Deportivo Alaves
3,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca,Alverca
4,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca,Alverca
...,...,...,...,...,...,...
597,/g/11h272dt50,/m/06b4z5,Paris FC,NaN,Paris FC,Paris FC
598,/g/11h272dt50,/m/06b4z5,Paris FC,NaN,Paris FC,Paris FC
599,/g/11h272dt50,/m/06b4z5,Paris FC,NaN,Paris FC,Paris FC
600,/g/11h272dt50,/m/08zfq_,Clermont Foot 63,NaN,Clermont,Clermont


# Dropping duplicate teams

In [ ]:
teams = teams.drop_duplicates()
teams.shape

# Simplify Texts

In [3]:
import re
import unicodedata

def simplify_name(text):
    # Ensure text is string type to handle potential float (NaN) values
    text = str(text)
    # 1. Remove quotes (and other specific punctuation if desired)
    text = re.sub(r'["\\]', '', text)

    # 2. Replace hyphens with spaces
    text = text.replace("-", " ")

    # 3. Normalize and remove accents (NFD + Mn category)
    normalized = unicodedata.normalize('NFD', text)
    simplified = "".join(c for c in normalized if unicodedata.category(c) != 'Mn')

    # 4. Clean up whitespace
    simplified = " ".join(simplified.split())

    return simplified

teams['team_name'] = teams['team_name'].apply(simplify_name)

# Scraping the team names from infobox wikipedia


In [4]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "InfoboxExtractor/1.0 (contact: example@mail.com)"
}

def search_wikipedia_url(team_name: str) -> str | None:
    """Use Wikipedia's search API to find the best matching football club URL."""
    search_url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": f"{team_name} football club",
        "format": "json",
        "srlimit": 1
    }

    response = requests.get(search_url, headers=HEADERS, params=params, timeout=10)

    if response.status_code != 200:
        return None

    results = response.json().get("query", {}).get("search", [])

    if not results:
        return None

    title = results[0]["title"]  # e.g. "Chelsea F.C."
    print(f"[SEARCH] Resolved '{team_name}' → '{title}'")
    return f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"


def get_team_name(team_name: str) -> str:
    """
    Extracts the infobox title from a football club's Wikipedia page.
    Uses the Wikipedia search API to resolve the correct article.
    """
    try:
        url = search_wikipedia_url(team_name)

        if not url:
            return f"No Wikipedia page found for '{team_name}'."

        print(f"[QUERY] Fetching: {url}")
        response = requests.get(url, headers=HEADERS, timeout=10)
        print(f"[STATUS] {response.status_code} for '{team_name}'")

        if response.status_code != 200:
            return f"HTTP Error: {response.status_code}"

        soup = BeautifulSoup(response.text, "html.parser")

        title_tag = soup.select_one("table.infobox caption.infobox-title.fn.org")
        if title_tag:
            result = title_tag.get_text(strip=True)
            print(f"[OUTPUT] Matched 'fn org' caption → {result}")
            return result

        title_tag = soup.select_one("table.infobox caption.infobox-title")
        if title_tag:
            result = title_tag.get_text(strip=True)
            print(f"[OUTPUT] Matched generic caption → {result}")
            return result

        title_tag = soup.select_one("table.infobox th.infobox-above")
        if title_tag:
            result = title_tag.get_text(strip=True)
            print(f"[OUTPUT] Matched 'infobox-above' th → {result}")
            return result

        print(f"[OUTPUT] No infobox title found for '{team_name}'")
        return "No infobox title found."

    except requests.exceptions.RequestException as e:
        print(f"[ERROR] Request failed for '{team_name}': {e}")
        return f"Request failed: {e}"


In [5]:
teams["testing_team_name"] = teams['team_name'].apply(get_team_name)

[SEARCH] Resolved 'Deportivo Alaves' → 'Deportivo Alavés'
[QUERY] Fetching: https://en.wikipedia.org/wiki/Deportivo_Alavés
[STATUS] 200 for 'Deportivo Alaves'
[OUTPUT] Matched 'fn org' caption → Deportivo Alavés
[SEARCH] Resolved 'F.C. Alverca' → 'F.C. Alverca'
[QUERY] Fetching: https://en.wikipedia.org/wiki/F.C._Alverca
[STATUS] 200 for 'F.C. Alverca'
[OUTPUT] Matched 'fn org' caption → Alverca
[SEARCH] Resolved 'Deportivo Alaves' → 'Deportivo Alavés'
[QUERY] Fetching: https://en.wikipedia.org/wiki/Deportivo_Alavés
[STATUS] 200 for 'Deportivo Alaves'
[OUTPUT] Matched 'fn org' caption → Deportivo Alavés
[SEARCH] Resolved 'F.C. Alverca' → 'F.C. Alverca'
[QUERY] Fetching: https://en.wikipedia.org/wiki/F.C._Alverca
[STATUS] 200 for 'F.C. Alverca'
[OUTPUT] Matched 'fn org' caption → Alverca
[SEARCH] Resolved 'F.C. Alverca' → 'F.C. Alverca'
[QUERY] Fetching: https://en.wikipedia.org/wiki/F.C._Alverca
[STATUS] 200 for 'F.C. Alverca'
[OUTPUT] Matched 'fn org' caption → Alverca
[SEARCH] Resolv

## If any missing team name

In [6]:
(teams["testing_team_name"] == "HTTP Error: 404").value_counts(), teams.shape

(testing_team_name
 False    602
 Name: count, dtype: int64,
 (602, 5))

In [7]:
teams.head()

,Root,team_mid,team_name,team_name[1],testing_team_name
0,/g/11nfsgpx7x,/m/03tcwm,Deportivo Alaves,NaN,Deportivo Alavés
1,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca
2,/g/11nfsgpx7x,/m/03tcwm,Deportivo Alaves,NaN,Deportivo Alavés
3,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca
4,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca


## Removing special letters from teams name

In [8]:
teams['simplified_testing_team_name'] = teams['testing_team_name'].apply(simplify_name)
teams[['testing_team_name', 'simplified_testing_team_name']]

,testing_team_name,simplified_testing_team_name
0,Deportivo Alavés,Deportivo Alaves
1,Alverca,Alverca
2,Deportivo Alavés,Deportivo Alaves
3,Alverca,Alverca
4,Alverca,Alverca
...,...,...
597,Paris FC,Paris FC
598,Paris FC,Paris FC
599,Paris FC,Paris FC
600,Clermont,Clermont


In [12]:
teams.head(10)

,Root,team_mid,team_name,team_name[1],testing_team_name,simplified_testing_team_name
0,/g/11nfsgpx7x,/m/03tcwm,Deportivo Alaves,NaN,Deportivo Alavés,Deportivo Alaves
1,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca,Alverca
2,/g/11nfsgpx7x,/m/03tcwm,Deportivo Alaves,NaN,Deportivo Alavés,Deportivo Alaves
3,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca,Alverca
4,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca,Alverca
5,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca,Alverca
6,/g/11nfsgpx7x,/m/03tcwm,Deportivo Alaves,NaN,Deportivo Alavés,Deportivo Alaves
7,/g/11g2tq1zns,/m/011f4kd4,Pafos FC,NaN,Pafos,Pafos
8,/g/11g2tq1zns,/m/011f4kd4,Pafos FC,NaN,Pafos,Pafos
9,/g/11g2tq1zns,/m/011f4kd4,Pafos FC,NaN,Pafos,Pafos


# **Athlete DataSet**

In [13]:
from google.colab import auth
import gspread
from google.auth import default

# Authenticate the user
auth.authenticate_user()

# Get credentials and authorize gspread
creds, _ = default()
gc = gspread.authorize(creds)

spreadsheet = gc.open('test_set_LMS')

worksheet = spreadsheet.worksheet('Iteration 6')
worksheet

<Worksheet 'Iteration 6' id:1737475733>

### Load Data into DataFrame

In [14]:
# Get all values from the worksheet
rows = worksheet.get_all_values()

# Convert to a DataFrame (using the first row as headers)
df = pd.DataFrame(rows[1:], columns=rows[0])

#Taking sample df

new_df = df[:]
print(new_df.shape)

(100, 17)


In [15]:
new_df.head()

,MID,Athlete Name,Getty Image link 1,Image 1,Getty Image link 2,Image 2,Getty Image link 3,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5,Accuracy,Best Image Found By Tool,Best Image by Raters,Correct Best Image Found by Tool,Correct Best Image Found by Tool
0,/g/11q9y7b81q,Alex Lebarbier,,,,,,,,,,,,,,,
1,/g/11qgqdpx0l,Emerson Mendes de Carvalho,,,,,,,,,,,,,,,
2,/g/11rd01qnw_,Francesco Dell'Aquila,,,,,,,,,,,,,,,
3,/g/11c72rk338,Bobur Abdikholikov,,,,,,,,,,,,,,,
4,/g/11cny08bf1,Rayan Raveloson,,,,,,,,,,,,,Yes,,


In [16]:
new_teams = teams.drop_duplicates()
new_teams.shape

(148, 6)

In [17]:
new_teams.head()

,Root,team_mid,team_name,team_name[1],testing_team_name,simplified_testing_team_name
0,/g/11nfsgpx7x,/m/03tcwm,Deportivo Alaves,NaN,Deportivo Alavés,Deportivo Alaves
1,/g/11nfsgpx7x,/m/026y7gr,F.C. Alverca,NaN,Alverca,Alverca
7,/g/11g2tq1zns,/m/011f4kd4,Pafos FC,NaN,Pafos,Pafos
11,/g/11rd01qnw_,/m/0fjjnc,US Citta di Pontedera,NaN,Pontedera,Pontedera
12,/g/11rd01qnw_,/m/07r78j,Torino FC,NaN,Torino,Torino
